<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **# MNPS Job Classification Evaluation — Per-Record Likelihood Scoring (0–5)**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> - Uses a **human baseline of 88–94%** first-pass accuracy (centered at 91%).
> - Produces a **0–5 likelihood score per record (likelihood_0_5)** for “how human-like” the model’s classification is.
> - **Preserves all v7.5.3 logic**: Drive mounting, prompts, model, post-processing, validation
> - Uses resource files (KSACs, KSAC similarity groups, salary, time to correct) that rarely change.
> - Lets you **swap in different job description + model batch files** each run

In [ ]:
#@title 1. Mount Google Drive & unzip Evaluation Resources

from google.colab import drive
from pathlib import Path
import zipfile
import os

# 1) Mount Google Drive
drive.mount('/content/drive')

# Folder in your Drive where results will be saved
DRIVE_BASE = Path("/content/drive/MyDrive/MNPS_Job_Classification_Eval")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print("Drive results folder:", DRIVE_BASE)

# 2) Paths in the Colab runtime
DATA_DIR = Path("/content")
RESOURCES_ZIP = DATA_DIR / "Evaluation Resources.zip"   # upload this file here
RESOURCES_DIR = DATA_DIR / "evaluation_resources"      # unzip target

print("Looking for:", RESOURCES_ZIP)

if RESOURCES_ZIP.exists():
    RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(RESOURCES_ZIP, 'r') as z:
        z.extractall(RESOURCES_DIR)
    print("Extracted Evaluation Resources to:", RESOURCES_DIR)
else:
    print("⚠️  'Evaluation Resources.zip' not found in /content.")
    print("    Please upload it to the Colab runtime and re-run this cell.")


In [ ]:
#@title 2. Imports & global configuration

import pandas as pd
import numpy as np
from pathlib import Path

# ========= HUMAN BASELINE (EDIT HERE IF NEEDED) =========
# Target human first-pass accuracy on a typical mixed batch
HUMAN_BASELINE_ACCURACY_LOW = 0.88
HUMAN_BASELINE_ACCURACY_HIGH = 0.94
HUMAN_BASELINE_ACCURACY = 0.91  # central point; used for calculations

HUMAN_ERROR_RATE = 1.0 - HUMAN_BASELINE_ACCURACY

# How we weight different factors (all 0–1, combined then scaled to 0–5)
SEVERITY_WEIGHTS = {
    "exact": 1.0,        # model role == human role
    "same_group": 0.7,   # different role, same KSAC similarity group
    "cross_group": 0.1,  # different KSAC similarity groups
}

WEIGHT_SEVERITY = 0.5
WEIGHT_SALARY   = 0.3
WEIGHT_TIME     = 0.2

# Above this gap, salary difference is treated as "maximal"
MAX_SALARY_GAP = 75_000.0  # dollars

# ========= FILE LOCATIONS (EDIT THESE NAMES PER RUN) =========
# DATA_DIR and RESOURCES_DIR are defined in Cell 1

FILES = {
    # BATCH-SPECIFIC INPUTS (change every run)
    # Upload these into /content and adjust names if needed.
    "sample_jds": DATA_DIR / "Sample JDs.csv",
    "batch_predictions": DATA_DIR / "Job_Classifications_Batch.csv",

    # MOSTLY-STABLE RESOURCES (come from Evaluation Resources.zip)
    "ksacs": RESOURCES_DIR / "MNPS KSACs.csv",
    "role_groups": RESOURCES_DIR / "MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv",
    "salary": RESOURCES_DIR / "salary_by_major_role_grouping.csv",
    "time_to_correct": RESOURCES_DIR / "Time to correct an error in hours.csv",
}

print("Configured file paths:")
for k, v in FILES.items():
    print(f"  {k}: {v}")


In [ ]:
#@title 3. CSV loader + load inputs

def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    """
    Try multiple encodings so you don't have to remember which one
    each file was saved with.
    """
    for enc in ("utf-8", "latin1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            continue
    # Last resort: let pandas guess
    return pd.read_csv(path, **kwargs)

# Load all resource files
df_jd   = read_csv_smart(FILES["sample_jds"])
df_pred = read_csv_smart(FILES["batch_predictions"])
df_ksacs = read_csv_smart(FILES["ksacs"])
df_groups = read_csv_smart(FILES["role_groups"])
df_salary = read_csv_smart(FILES["salary"])
df_time   = read_csv_smart(FILES["time_to_correct"])

print("Sample JDs shape:           ", df_jd.shape)
print("Model batch predictions:    ", df_pred.shape)
print("KSACs table shape:          ", df_ksacs.shape)
print("KSAC similarity groups:     ", df_groups.shape)
print("Salary table shape:         ", df_salary.shape)
print("Time-to-correct table:      ", df_time.shape)


In [ ]:
#@title 4. Lookup tables: role → KSAC group, role → salary, time stats

# ---- KSAC similarity groups: Role → Group Name ----
role_to_group = {}
for _, row in df_groups.iterrows():
    group_name = row["Group Name"]
    roles = [r.strip() for r in str(row["Roles in Group"]).split(",")]
    for r in roles:
        if r:
            role_to_group[r] = group_name

# ---- Salary table: clean $, commas, spaces ----
def clean_money(val):
    if isinstance(val, str):
        val = val.replace("$", "").replace(",", "").strip()
        if not val:
            return np.nan
        try:
            return float(val)
        except ValueError:
            return np.nan
    try:
        return float(val)
    except Exception:
        return np.nan

df_salary_clean = df_salary.copy()
df_salary_clean.columns = [c.strip() for c in df_salary_clean.columns]

for col in ["Min Annual Salary", "Average Annual Salary", "Max Annual Salary"]:
    df_salary_clean[col] = df_salary_clean[col].apply(clean_money)

def get_salary_stats(role: str):
    """
    Return (min, avg, max) annual salary for a given role, or NaNs if missing.
    """
    if not isinstance(role, str):
        return (np.nan, np.nan, np.nan)
    row = df_salary_clean[df_salary_clean["Major Role Grouping"] == role]
    if row.empty:
        return (np.nan, np.nan, np.nan)
    r = row.iloc[0]
    return (r["Min Annual Salary"], r["Average Annual Salary"], r["Max Annual Salary"])

# ---- Time-to-correct stats ----
time_min = float(df_time["Min"].iloc[0])
time_avg = float(df_time["Average"].iloc[0])
time_max = float(df_time["Max"].iloc[0])

baseline_time_per_job = HUMAN_ERROR_RATE * time_avg

print("Example salary rows:")
display(df_salary_clean.head(3))

print("\nTime to correct (hours):")
print(f"  Min: {time_min}, Avg: {time_avg}, Max: {time_max}")
print(f"Baseline human time/job (given {HUMAN_BASELINE_ACCURACY:.0%} accuracy): {baseline_time_per_job:.2f} hours")


In [ ]:
#@title 5. Infer baseline human role from Job Description Name

def infer_role_from_title(title: str) -> str:
    """
    Heuristic mapping from MNPS job description name → canonical MNPS role.
    Adjust this function if you refine your naming patterns.
    """
    if not isinstance(title, str) or not title.strip():
        return None
    t = title.lower()

    # Most specific patterns first
    if "assistant principal" in t or ("principal" in t and "asst" in t):
        return "Assistant Principal"
    if "principal" in t:
        return "Principal"

    if "teacher" in t and "rotc" in t:
        return "Instructor"
    if t.startswith("teacher") or " teacher" in t:
        return "Teacher"

    if t.startswith("coach") or " coach" in t:
        return "Coach"

    if t.startswith("coord") or " coord" in t:
        return "Coordinator"

    if t.startswith("analyst") or " analyst" in t:
        return "Analyst"

    if t.startswith("accountant") or " accountant" in t:
        return "Accountant"

    if t.startswith("spec") or " specialist" in t:
        return "Specialist"

    if "social worker" in t:
        return "Social Worker"
    if "counselor" in t:
        return "Counselor"
    if "librarian" in t:
        return "Librarian"
    if "therapist" in t:
        return "Therapist"
    if "translator" in t:
        return "Translator"
    if "clerk" in t:
        return "Clerk"

    if "admin " in t and "records" in t:
        return "Administrative Assistant"

    if "rep " in t or t.startswith("rep"):
        return "Representative"

    if "skilled laborer" in t:
        return "Skilled Laborer"

    if "intern" in t:
        return "Intern"

    if "tech " in t or t.startswith("tech"):
        return "Technician"

    if t.startswith("asst "):
        if "dir " in t or "director" in t:
            return "Assistant"
        return "Assistant"

    if t.startswith("dir ") or " director" in t or t.startswith("director"):
        return "Director"

    if "manager" in t or t.startswith("mgr"):
        return "Manager"

    # Fallback: if any known role name appears in the title, use it
    for role in sorted(role_to_group.keys(), key=len, reverse=True):
        if role.lower() in t:
            return role

    return None

def get_similarity_group(role: str) -> str:
    if not isinstance(role, str):
        return None
    return role_to_group.get(role)

print("Sample inferred roles:")
sample_titles = df_jd["Job Description Name"].head(10).tolist()
for t in sample_titles:
    print(f"{t:45s} → {infer_role_from_title(t)}")


In [ ]:
#@title 6. Merge Sample JDs with model predictions

# Ensure JDs have a stable 0-based index to match `source_row_index`
df_jd_reset = df_jd.reset_index().rename(columns={"index": "source_row_index"})

# Merge with model predictions
df_merged = df_jd_reset.merge(
    df_pred,
    on="source_row_index",
    how="inner",
    validate="one_to_one"
)

# Human vs model roles
df_merged["human_role"] = df_merged["Job Description Name"].apply(infer_role_from_title)
df_merged["model_role"] = df_merged["major_role_group"]  # change if your column name differs

print("Merged shape:", df_merged.shape)
df_merged[["source_row_index", "Job Description Name", "human_role", "model_role"]].head(10)


In [ ]:
#@title 7. Compute severity, cost, and 0–5 likelihood per record

def compute_severity(model_role: str, human_role: str) -> str:
    """
    Classify the mismatch severity:
    - 'exact'       : same role label
    - 'same_group'  : roles differ but share KSAC similarity group
    - 'cross_group' : roles fall in different similarity groups
    - 'unknown'     : can't determine
    """
    if not isinstance(model_role, str) or not isinstance(human_role, str):
        return "unknown"
    if model_role == human_role:
        return "exact"

    g_model = get_similarity_group(model_role)
    g_human = get_similarity_group(human_role)

    if g_model is not None and g_model == g_human:
        return "same_group"
    return "cross_group"

def severity_to_time(severity: str) -> float:
    """
    Map severity to an approximate correction time in hours.
    """
    if severity == "exact":
        return 0.0
    elif severity == "same_group":
        return time_min          # light edit / review
    elif severity == "cross_group":
        return time_max          # heavy rework
    else:
        return time_avg          # unknown → mid

def compute_record_scores(model_role: str, human_role: str):
    """
    Core scoring logic for one record. Returns a dict including:
      - severity
      - severity_score   (0–1)
      - salary_gap       (absolute $ difference)
      - salary_score     (0–1)
      - time_hours       (estimated hours)
      - time_score       (0–1)
      - likelihood_0_5   (0–5, final metric)
    """
    severity = compute_severity(model_role, human_role)
    severity_score = SEVERITY_WEIGHTS.get(severity, 0.3)

    # Salary difference
    _, human_avg, _ = get_salary_stats(human_role)
    _, model_avg, _ = get_salary_stats(model_role)

    if np.isnan(human_avg) or np.isnan(model_avg):
        salary_gap = 0.0
    else:
        salary_gap = abs(model_avg - human_avg)

    salary_norm = min(1.0, salary_gap / MAX_SALARY_GAP)  # 0 = no gap, 1 = >= MAX_SALARY_GAP
    salary_score = 1.0 - salary_norm                      # 1 = perfect, 0 = large gap

    # Time-to-correct
    t = severity_to_time(severity)
    if baseline_time_per_job <= 0:
        time_score = 1.0
    else:
        time_ratio = t / baseline_time_per_job
        time_score = 1.0 / (1.0 + time_ratio)  # more hours → lower score

    combined_0_1 = (
        WEIGHT_SEVERITY * severity_score +
        WEIGHT_SALARY   * salary_score   +
        WEIGHT_TIME     * time_score
    )
    combined_0_1 = max(0.0, min(1.0, combined_0_1))
    likelihood_0_5 = 5.0 * combined_0_1

    return {
        "severity": severity,
        "severity_score": severity_score,
        "salary_gap": salary_gap,
        "salary_score": salary_score,
        "time_hours": t,
        "time_score": time_score,
        "likelihood_0_5": likelihood_0_5,
    }

score_cols = [
    "severity", "severity_score",
    "salary_gap", "salary_score",
    "time_hours", "time_score",
    "likelihood_0_5",
]

scores = df_merged.apply(
    lambda row: pd.Series(
        compute_record_scores(row["model_role"], row["human_role"])
    ),
    axis=1
)

df_scored = pd.concat([df_merged, scores], axis=1)

print("Per-record likelihood (0–5) sample:")
df_scored[
    ["source_row_index", "Job Description Name",
     "human_role", "model_role", "severity", "likelihood_0_5"]
].head(10)


In [ ]:
#@title 8. Batch summary statistics

severity_counts = df_scored["severity"].value_counts(dropna=False).rename("count")
severity_pct = (severity_counts / len(df_scored) * 100).round(1).rename("percent")

summary_severity = pd.concat([severity_counts, severity_pct], axis=1)

print("Severity breakdown:")
display(summary_severity)

print("\nLikelihood (0–5) distribution:")
display(df_scored["likelihood_0_5"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

overall_mean = df_scored["likelihood_0_5"].mean()
print(f"\nAverage per-record likelihood (0–5): {overall_mean:.2f}")

print(
    f"\nInterpretation guide (given {HUMAN_BASELINE_ACCURACY_LOW:.0%}–"
    f"{HUMAN_BASELINE_ACCURACY_HIGH:.0%} human baseline):\n"
    "  ~4.0–5.0  → Very likely to match a typical human classifier\n"
    "  ~2.5–4.0  → Borderline / needs human review\n"
    "  < 2.5     → Unlikely to match human judgement; treat as high-risk\n"
)


In [ ]:
#@title 9. Export per-record scores to CSV (local + Drive)

# You can rename this per run if you want
output_filename = "Job_Classifications_Batch_scored.csv"

output_path_local = DATA_DIR / output_filename
output_path_drive = DRIVE_BASE / output_filename

df_scored.to_csv(output_path_local, index=False)
df_scored.to_csv(output_path_drive, index=False)

print("Saved per-record scored output (local):", output_path_local)
print("Saved per-record scored output (Drive):", output_path_drive)

df_scored.head(5)
